# 02 · Evaluate the distraction models
Final, honest numbers on the **test set** — 4 drivers that were never used for training *or* for choosing models.

1. Test-set metrics for all three models (10-class + safe-vs-distracted)
2. Temperature scaling (confidence calibration), fitted on validation only
3. Model selection — decided on **validation**, then reported on test
4. Confusion matrix and per-class results
5. Grad-CAM: what the model looks at
6. Worst mistakes
7. Save results and figures to Drive (for the README)

**Runtime:** T4 GPU (CPU works too, just slower — about 10 min).

## 1 · Settings

In [ ]:
GITHUB_USER = "harshithdande28"
REPO_NAME = "driver-monitoring-system"
MODELS = ["baseline", "efficientnet_b0", "mobilenet_v3_large"]

## 2 · Mount Drive and get the latest code

In [ ]:
import os, sys, torch
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = f"/content/{REPO_NAME}"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 3 · Load config, data and the split

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, class_names
from src.data.prepare import extract_dataset
from src.data.split import check_no_leakage
from src.data.dataset import build_dataloaders
from src.models.factory import build_model
from src.training.checkpoint import load_checkpoint
from src.training.train import evaluate
from src.evaluation.metrics import (softmax, multiclass_metrics, binary_metrics, per_class_report,
                                    plot_confusion_matrix, plot_reliability)
from src.evaluation.calibration import fit_temperature, apply_temperature
from src.evaluation.gradcam import plot_gradcam_grid

cfg = load_config()
CLASS_NAMES = class_names(cfg)
CLASS_CODES = list(CLASS_NAMES)
TCFG = cfg["train"]
DRIVE_ROOT = Path(cfg["paths"]["drive_root"])
CKPT_ROOT = DRIVE_ROOT / "checkpoints"
RESULTS = DRIVE_ROOT / "results"
FIGS = RESULTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

LOCAL_DATA = extract_dataset(DRIVE_ROOT / "data" / cfg["data"]["zip_name"], cfg["paths"]["local_data"])
split_df = pd.read_csv(cfg["paths"]["splits_file"])
check_no_leakage(split_df)
loaders = build_dataloaders(split_df, LOCAL_DATA, CLASS_CODES, image_size=TCFG["image_size"],
                            batch_size=TCFG["batch_size"], num_workers=TCFG["num_workers"], seed=cfg["seed"])
test_df = loaders["test"].dataset.df          # same order as the test predictions
print({k: len(v.dataset) for k, v in loaders.items()}, "images")
print("Test drivers:", sorted(test_df["subject"].unique()))

## 4 · Run every model on validation and test (~1–2 min per model)

In [ ]:
def load_model(name):
    ckpt = load_checkpoint(CKPT_ROOT / name / "best.pt", DEVICE)
    if ckpt is None:
        raise FileNotFoundError(f"No best.pt for {name} — train it in 01_train_distraction first.")
    model = build_model(name, len(CLASS_CODES), pretrained=False)
    model.load_state_dict(ckpt["model"])
    return model.to(DEVICE).eval(), ckpt

outputs = {}
for name in MODELS:
    model, ckpt = load_model(name)
    val = evaluate(model, loaders["val"], DEVICE)
    test = evaluate(model, loaders["test"], DEVICE)
    outputs[name] = {
        "val_logits": val["logits"].numpy(), "val_labels": val["labels"].numpy(),
        "test_logits": test["logits"].numpy(), "test_labels": test["labels"].numpy(),
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
        "best_epoch": ckpt["epoch"],
    }
    print(f"{name:20s} val acc {val['acc']:.3f} | test acc {test['acc']:.3f}")
    del model
    torch.cuda.empty_cache()

## 5 · Temperature scaling (calibration)
T is fitted on **validation** and then applied to test. It never changes the predicted class —
it only fixes over/under-confidence, which the alert logic depends on.
**ECE** (expected calibration error) = average gap between confidence and actual accuracy; lower is better.

In [ ]:
temperatures = {}
for name, o in outputs.items():
    T = fit_temperature(o["val_logits"], o["val_labels"])
    temperatures[name] = T
    o["val_probs"] = apply_temperature(o["val_logits"], T)
    o["test_probs_raw"] = softmax(o["test_logits"])
    o["test_probs"] = apply_temperature(o["test_logits"], T)
    print(f"{name:20s} T = {T:.3f}")

(RESULTS / "temperatures.json").write_text(json.dumps(temperatures, indent=2))

## 6 · Results table

In [ ]:
rows = []
for name, o in outputs.items():
    val_m = multiclass_metrics(o["val_probs"], o["val_labels"])
    raw = multiclass_metrics(o["test_probs_raw"], o["test_labels"])
    cal = multiclass_metrics(o["test_probs"], o["test_labels"])
    binm = binary_metrics(o["test_probs"], o["test_labels"])
    rows.append({
        "model": name, "params_M": round(o["params_M"], 2), "best_epoch": o["best_epoch"],
        "val_macro_f1": val_m["macro_f1"],
        "test_acc": cal["accuracy"], "test_macro_f1": cal["macro_f1"],
        "test_log_loss": cal["log_loss"],
        "ece_before": raw["ece"], "ece_after": cal["ece"],
        "safe_vs_distracted_f1": binm["bin_f1"], "distraction_recall": binm["bin_recall"],
        "false_alarm_rate": binm["false_alarm_rate"],
    })
results = pd.DataFrame(rows).set_index("model").round(3)
results.to_csv(RESULTS / "test_results.csv")
results

## 7 · Choose the model — by validation, not test
Picking the model with the best *test* score would quietly turn the test set into a validation set.
So the choice uses **validation macro-F1**, and the test numbers above are reported as they are.

In [ ]:
CHOSEN = results["val_macro_f1"].idxmax()
print("Chosen model:", CHOSEN)
print(results.loc[CHOSEN, ["test_acc", "test_macro_f1", "ece_after", "false_alarm_rate"]])
(RESULTS / "chosen_model.json").write_text(json.dumps(
    {"model": CHOSEN, "temperature": temperatures[CHOSEN]}, indent=2))

## 8 · Calibration before vs after

In [ ]:
o = outputs[CHOSEN]
fig = plot_reliability(o["test_probs_raw"], o["test_probs"], o["test_labels"], title=f"{CHOSEN} — test set")
fig.savefig(FIGS / "reliability.png", dpi=150); plt.show()

## 9 · Confusion matrix and per-class results (test set)

In [ ]:
fig = plot_confusion_matrix(o["test_probs"], o["test_labels"], CLASS_NAMES, title=f"{CHOSEN} — test set")
fig.savefig(FIGS / "confusion_matrix.png", dpi=150); plt.show()

report = per_class_report(o["test_probs"], o["test_labels"], CLASS_NAMES)
report.to_csv(RESULTS / "per_class_test.csv", index=False)
report

## 10 · Grad-CAM — what the model looks at
Red = regions that most influenced the prediction. Good signs: hands, phone, cup, head, mirror/radio area.
Bad signs: the seat, the window, or the car interior far from the driver.

In [ ]:
model, _ = load_model(CHOSEN)
test_ds = loaders["test"].dataset
rng = np.random.default_rng(cfg["seed"])
samples = [int(rng.choice(np.where(o["test_labels"] == c)[0])) for c in range(len(CLASS_CODES))]
fig = plot_gradcam_grid(model, test_ds, samples, CLASS_NAMES, ncols=5)
fig.savefig(FIGS / "gradcam_per_class.png", dpi=150); plt.show()

## 11 · Worst mistakes (confident and wrong)

In [ ]:
preds = o["test_probs"].argmax(1)
conf = o["test_probs"].max(1)
wrong = np.where(preds != o["test_labels"])[0]
worst = wrong[np.argsort(-conf[wrong])][:10]
fig = plot_gradcam_grid(model, test_ds, list(worst), CLASS_NAMES, ncols=5)
fig.savefig(FIGS / "gradcam_worst_errors.png", dpi=150); plt.show()

pd.DataFrame({
    "driver": test_df.loc[worst, "subject"].values,
    "true": [CLASS_CODES[i] for i in o["test_labels"][worst]],
    "pred": [CLASS_CODES[i] for i in preds[worst]],
    "confidence": conf[worst].round(3),
    "path": test_df.loc[worst, "path"].values,
})

## 12 · Accuracy per test driver

In [ ]:
per_driver = (pd.DataFrame({"driver": test_df["subject"].values,
                            "correct": preds == o["test_labels"]})
                .groupby("driver")["correct"].agg(["mean", "count"])
                .rename(columns={"mean": "accuracy", "count": "images"}).round(3))
per_driver.to_csv(RESULTS / "per_driver_test.csv")
per_driver

---
✅ **Done.** Saved to Drive → `driver-monitoring/results/`:
`test_results.csv`, `per_class_test.csv`, `per_driver_test.csv`, `temperatures.json`, `chosen_model.json`, and `figures/` (for the README).

**Next:** the drowsiness module on your laptop, then ONNX export of the chosen model.